# Lab | Agent & Vector store


<br>

## Intro

In this lab you'll build an AI agent that knows when to consult *different* knowledge bases to answer a question — instead of relying on a single source of truth.

Here's what to expect:

1. **Follow a full worked demo** — We'll walk through every step together: ingesting the *state of the union* speech and the *Ruff* docs into two vector stores, wrapping each in a `RetrievalQA` tool, and building an agent that picks the right tool (or both!) depending on the question.

2. **Replicate it yourself with a new dataset** — Then, you'll swap in a dataset of your choice and rebuild the same pipeline, adapting the prompts and tools along the way.

By the end of this lab, you'll understand how to build multi-source AI agents and be able to apply the pattern to your own datasets.

<br>

## Combine agents and vector stores

Let's get into the demo. We'll wrap each vector store in a `RetrievalQA` chain and hand it to an agent as a `Tool`. The agent then decides, at each step, which tool to call based purely on its description — this is what lets it route between multiple knowledge sources.

There are two flavors of this pattern, both of which we'll try below:

- **Agent as reasoner** — the agent calls a tool and can keep reasoning afterward (e.g. to combine results from multiple sources).
- **Agent as router** (`return_direct=True`) — the agent just picks the right tool and returns its answer immediately, no extra reasoning.

<br>

## Install dependencies

Uncomment and run the cells below to install the dependencies required for this notebook.

Tip: Use a virtual environment to keep this project's dependencies isolated from your system Python and other projects.

# NOTE TO ME: Why do I need all the packages to download?

 # 1 - langchain, langchain-core, langchain-community, langchain-openai, langchain-text-splitters: the notebook’s chains, tools, OpenAI integration, and text splitting.
 # 2 - chromadb: stores the document embeddings for retrieval.
 # 3 - python-dotenv: loads your OpenAI API key from a .env file.
 # 4 - beautifulsoup4: helps load and parse the Ruff webpage.
 # 5 - ipykernel: connects a Python environment to Jupyter/VS Code

In [6]:
%pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2" "langchain-text-splitters<0.3" python-dotenv==1.2.2 chromadb==1.5.9 beautifulsoup4==4.15.0 ipykernel

  Using cached langchain-0.2.17-py3-none-any.whl.metadata (7.1 kB)
  Using cached langchain_core-0.2.43-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain_community-0.2.19-py3-none-any.whl.metadata (2.7 kB)
  Using cached langchain_openai-0.1.25-py3-none-any.whl.metadata (2.6 kB)
  Using cached langchain_text_splitters-0.2.4-py3-none-any.whl.metadata (2.3 kB)
  Using cached beautifulsoup4-4.15.0-py3-none-any.whl.metadata (3.8 kB)
     ---------------------------------------- 0.0/137.6 kB ? eta -:--:--
     ---------------------------- --------- 102.4/137.6 kB 6.1 MB/s eta 0:00:01
     -------------------------------------- 137.6/137.6 kB 2.1 MB/s eta 0:00:00
  Using cached uvicorn-0.54.0-py3-none-any.whl.metadata (6.6 kB)
     ---------------------------------------- 0.0/52.0 kB ? eta -:--:--
     ---------------------------------------- 52.0/52.0 kB 2.8 MB/s eta 0:00:00
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached typer-0.27.3-py3-none-any.whl.m

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

In [20]:
# !pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2"

# NOTE TO ME: Why do I need lang chain for building multi-source agents? 

Building agents that pull from multiple data sources and vector stores requires standardizing unstructured documents, managing chunking strategies, embedding texts, and routing queries across different backends. LangChain provides pre-built integrations and composable components that streamline this data pipeline. 

LangChain abstracts away vendor-specific APIs for document loaders, text splitters, embedding models, and vector stores into a single interface. This allows you to swap out vector databases (such as Pinecone, Chroma, Milvus, or Postgres) or add new data sources (PDFs, S3 buckets, SQL databases) by changing just a few lines of configuration rather than rewriting your core retrieval logic.


In [21]:
# !pip install python-dotenv==1.2.2 chromadb==1.5.9 beautifulsoup4==4.15.0

<br>

## Initial Setup

Before building anything, we need to load our API credentials, instantiate the LLM we'll use throughout the notebook, and locate the sample document we'll be querying.

In [22]:
from langchain.chains import RetrievalQA
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader

In [23]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')


In [24]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path.cwd() / ".env"
print("Notebook is looking in:", Path.cwd())
print(".env file found:", env_path.is_file())

load_dotenv(env_path, override=True)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print("API key loaded:", bool(OPENAI_API_KEY))

Notebook is looking in: c:\Users\aysea\OneDrive\Desktop\Education & Professional Development\AI & Technology Education\Ironhack AI Engineering Bootcamp Program\lab-agent-vector-store
.env file found: True
API key loaded: True


In [25]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=OPENAI_API_KEY)

<br>

Now let's ingest the state of the union speech: load the raw text, split it into manageable chunks, embed those chunks, and store them in a Chroma vector store.

In [26]:
doc_path =  "./datasets/state_of_the_union.txt"

In [27]:
loader = TextLoader(doc_path, encoding="utf-8")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)

embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)

docsearch = Chroma.from_documents(texts, embeddings, collection_name="state-of-union")

<br>

## Adding a second knowledge source

To show how an agent can route between multiple tools, let's add a second vector store — this time built from the Ruff FAQ web page instead of a local file.

In [28]:
state_of_union = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=docsearch.as_retriever()
)

In [29]:
from langchain_community.document_loaders import WebBaseLoader

In [30]:
loader = WebBaseLoader("https://beta.ruff.rs/docs/faq/")

In [31]:
docs = loader.load()
ruff_texts = text_splitter.split_documents(docs)
ruff_db = Chroma.from_documents(ruff_texts, embeddings, collection_name="ruff")
ruff = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=ruff_db.as_retriever()
)

Created a chunk of size 2209, which is longer than the specified 1000
Created a chunk of size 3190, which is longer than the specified 1000
Created a chunk of size 1017, which is longer than the specified 1000
Created a chunk of size 2313, which is longer than the specified 1000


<br>

## Create the Agent

With both `RetrievalQA` chains ready, we wrap each one in a `Tool` (giving it a name and a description the agent will use to decide when to call it), then hand both tools to an agent.

In [32]:
# Import things that are needed generically
from langchain.agents import AgentType, Tool, initialize_agent
from langchain_openai import OpenAI

<br>

Let's try it out — first with a question only the state of the union tool can answer, then one only Ruff can answer. Watch the verbose output to see which tool the agent picks each time.

In [33]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
    ),
]

In [34]:
# Construct the agent. We will use the default agent type here.
# See documentation for a full list of options.
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [36]:
agent.invoke(
    "What did biden say about ketanji brown jackson in the state of the union address?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to find out what President Biden said about Ketanji Brown Jackson in the most recent State of the Union address. 
Action: State of Union QA System
Action Input: What did Biden say about Ketanji Brown Jackson in the State of the Union address?
Observation: In the State of the Union address, Biden mentioned that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson four days prior to the address. He described her as one of the nation's top legal minds who will continue Justice Stephen Breyer's legacy of excellence.
Thought:I now know the final answer.  
Final Answer: In the State of the Union address, President Biden mentioned that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson four days prior to the address, describing her as one of the nation's top legal minds who will continue Justice Stephen Breyer's legacy of excellence.

> Finished chain.


{'input': 'What did biden say about ketanji brown jackson in the state of the union address?',
 'output': "In the State of the Union address, President Biden mentioned that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson four days prior to the address, describing her as one of the nation's top legal minds who will continue Justice Stephen Breyer's legacy of excellence."}

In [37]:
agent.invoke("Why use ruff over flake8?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to gather information about the advantages of using ruff compared to flake8, which is a popular Python linter. This will help me provide a comprehensive answer to the question.  
Action: Ruff QA System  
Action Input: "What are the advantages of using ruff over flake8?"  
Observation: The advantages of using Ruff over Flake8 include:

1. **Performance**: Ruff is implemented in Rust, which generally allows it to run faster than Flake8, which is written in Python.

2. **More Rules**: Ruff implements over 900 rules, while Flake8 has a smaller set of rules. This means Ruff can catch a wider variety of linting issues.

3. **Automatic Fixes**: Ruff is capable of automatically fixing its own lint violations, whereas Flake8 does not have this capability.

4. **No Custom Rules**: Ruff does not support custom lint rules, but it re-implements popular Flake8 plugins natively, which can simplify the setup and reduce conflicts.

5. **Simplified Rule Selection**: Ruff uses different rule codes

{'input': 'Why use ruff over flake8?',
 'output': 'The advantages of using Ruff over Flake8 include better performance due to its Rust implementation, a larger set of over 900 rules, the ability to automatically fix lint violations, simplified rule selection, independence from plugins, and support for multiple tools. Overall, Ruff offers a more comprehensive and efficient linting experience compared to Flake8.'}

# NOTE TO ME: The output shows the agent doing what the lab intends:

Thought: it decided it needed the state of the union speech.
Action: it picked the State of Union QA System tool, which is the right one.
Observation: that tool searched the vector store and returned the passage about Biden nominating Judge Ketanji Brown Jackson.
Final Answer: the agent wrote a final answer from that result.
This is the "agent as reasoner" version, where the agent keeps thinking after the tool call.

Keep running the notebook in order:

Run agent.invoke("Why use ruff over flake8?") and check that it picks the Ruff tool.
In the return_direct=True section, rerun the tools and agent cells first, then the two questions. The agent should return the tool's answer straight away, without a final "Thought".
Run the multi-hop question about Ruff and Jupyter. It should use both tools.

## Use the Agent solely as a router

<br>

You can also set `return_direct=True` if you intend to use the agent as a router and just want to directly return the result of the RetrievalQAChain.

Notice that in the above examples the agent did some extra work after querying the RetrievalQAChain. You can avoid that and just return the result directly.

In [38]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question.",
        return_direct=True,
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
        return_direct=True,
    ),
]

In [39]:
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [40]:
agent.invoke(
    "What did biden say about ketanji brown jackson in the state of the union address?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to find out what President Biden said about Ketanji Brown Jackson in the most recent State of the Union address. 
Action: State of Union QA System
Action Input: What did Biden say about Ketanji Brown Jackson in the State of the Union address?
Observation: In the State of the Union address, Biden mentioned that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson four days prior to the address. He referred to her as one of the nation's top legal minds and stated that she would continue Justice Stephen Breyer's legacy of excellence.


> Finished chain.


{'input': 'What did biden say about ketanji brown jackson in the state of the union address?',
 'output': "In the State of the Union address, Biden mentioned that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson four days prior to the address. He referred to her as one of the nation's top legal minds and stated that she would continue Justice Stephen Breyer's legacy of excellence."}

In [41]:
agent.invoke("Why use ruff over flake8?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to compare the features and advantages of Ruff and Flake8 to provide a comprehensive answer about why one might choose Ruff over Flake8. 
Action: Ruff QA System
Action Input: "Why should someone use Ruff over Flake8 for Python linting?"
Observation: Someone might choose to use Ruff over Flake8 for Python linting for several reasons:

1. **Performance**: Ruff is implemented in Rust, which generally allows it to run faster than Flake8, providing quicker feedback on lint violations.

2. **Comprehensive Rule Set**: Ruff implements over 900 rules, significantly more than Flake8, which allows for more extensive linting coverage.

3. **Automatic Fixing**: Unlike Flake8, Ruff is capable of automatically fixing its own lint violations, which can save time during development.

4. **No Custom Rules Needed**: Ruff does not support custom lint rules, but it re-implements popular Flake8 plugins natively, reducing the need for additional plugins and potential conflicts.

5. **Simplified Rule S

{'input': 'Why use ruff over flake8?',
 'output': 'Someone might choose to use Ruff over Flake8 for Python linting for several reasons:\n\n1. **Performance**: Ruff is implemented in Rust, which generally allows it to run faster than Flake8, providing quicker feedback on lint violations.\n\n2. **Comprehensive Rule Set**: Ruff implements over 900 rules, significantly more than Flake8, which allows for more extensive linting coverage.\n\n3. **Automatic Fixing**: Unlike Flake8, Ruff is capable of automatically fixing its own lint violations, which can save time during development.\n\n4. **No Custom Rules Needed**: Ruff does not support custom lint rules, but it re-implements popular Flake8 plugins natively, reducing the need for additional plugins and potential conflicts.\n\n5. **Simplified Rule Selection**: Ruff uses different rule codes and prefixes to minimize conflicts across plugins, allowing users to toggle rules more easily.\n\n6. **Independence from Plugins**: Ruff implements all r

<br>

## Multi-Hop vector store reasoning

Because vector stores are easily usable as tools in agents, it is easy to use answer multi-hop questions that depend on vector stores using the existing agent framework.

In [42]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
]

In [43]:
# Construct the agent. We will use the default agent type here.
# See documentation for a full list of options.
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [44]:
agent.invoke(
    "What tool does ruff use to run over Jupyter Notebooks? Did the president mention that tool in the state of the union?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to find out what tool Ruff uses to run over Jupyter Notebooks and whether the president mentioned that tool in the State of the Union address. I will first check the Ruff QA System for information about the tool used by Ruff.

Action: Ruff QA System  
Action Input: What tool does Ruff use to run over Jupyter Notebooks?  
Observation: Ruff integrates with nbQA, a tool for running linters and code formatters over Jupyter Notebooks.
Thought:Now that I know Ruff uses nbQA to run over Jupyter Notebooks, I need to check if the president mentioned nbQA in the State of the Union address.

Action: State of Union QA System  
Action Input: Did the president mention nbQA in the State of the Union address?  
Observation: I don't know.
Thought:I need to find out if the president mentioned nbQA in the State of the Union address. Since I don't have the information yet, I will attempt to gather it again.

Action: State of Union QA System  
Action Input: Did the president mention any tools relate

{'input': 'What tool does ruff use to run over Jupyter Notebooks? Did the president mention that tool in the state of the union?',
 'output': 'Ruff uses nbQA to run over Jupyter Notebooks, but I do not have information on whether the president mentioned nbQA or any related tools in the State of the Union address.'}

<br><br>

---

<br>

## 🚀 Your turn

Time to make this lab your own! 

Replace the `state_of_the_union.txt` dataset with something you'd actually enjoy chatting with. A great place to start is the [sonnets.txt dataset](https://github.com/martin-gorner/tensorflow-rnn-shakespeare/blob/master/shakespeare/sonnets.txt) —or any other .txt file from that repository. Of course, you're not limited to those options. Pick any text that interests you and see how your chatbot responds.

Here's what to do:

1. **Get your data** — Download your chosen `.txt` file into this project folder (or point `TextLoader` at it directly).
2. **Rebuild the vector store** — Load, split, and embed your new document, then create a fresh `RetrievalQA` chain for it (give it a descriptive `collection_name`!).
3. **Rewrite the tool description** — Update the `Tool`'s `name` and `description` so the agent knows *when* it should reach for this new tool instead of the Ruff or state-of-the-union ones.
4. **Rebuild the agent** — Combine your new tool with the existing Ruff tool (or drop it if you'd rather keep just your new dataset + one other source).
5. **Put it to the test** — Ask your agent:
   - A direct question that only your new dataset can answer.
   - A question that only the Ruff tool can answer.
   - A multi-hop question that requires combining *both* tools' knowledge, like the Jupyter/Ruff example above.
6. **Reflect** — In a markdown cell, briefly note whether the agent picked the right tool(s) each time, and what happened when you set `return_direct=True` vs. not.

⭐️ **Bonus points:**
- Instead of modifying this same file, create a new file `solution.ipynb` and replicate the process from scratch.

💡 **Tip:** 
- Watch the `verbose=True` agent logs closely — they show you the agent's reasoning step by step, which is the best way to understand *why* it picked a particular tool.



## My dataset: Schopenhauer, *The Wisdom of Life*

I replaced the state of the union speech with Arthur Schopenhauer's essay collection *The Wisdom of Life* (public domain, from Project Gutenberg), saved as `datasets/schopenhauer_studies_in_pessimism.txt`. I combine it with the existing Ruff tool.

In [45]:
schopenhauer_path = "./datasets/schopenhauer_studies_in_pessimism.txt"

schop_documents = TextLoader(schopenhauer_path, encoding="utf-8").load()

# Drop the Project Gutenberg license header/footer so only the book is embedded
raw = schop_documents[0].page_content
start = raw.find("*** START OF")
end = raw.find("*** END OF")
if start != -1:
    raw = raw[raw.find("\n", start) + 1 : end if end != -1 else None]
schop_documents[0].page_content = raw

schop_texts = text_splitter.split_documents(schop_documents)
print(f"{len(schop_texts)} chunks")

schop_db = Chroma.from_documents(schop_texts, embeddings, collection_name="schopenhauer")
schopenhauer = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=schop_db.as_retriever()
)

Created a chunk of size 1080, which is longer than the specified 1000
Created a chunk of size 2649, which is longer than the specified 1000
Created a chunk of size 2688, which is longer than the specified 1000
Created a chunk of size 1993, which is longer than the specified 1000
Created a chunk of size 1699, which is longer than the specified 1000
Created a chunk of size 1745, which is longer than the specified 1000
Created a chunk of size 2018, which is longer than the specified 1000
Created a chunk of size 1211, which is longer than the specified 1000
Created a chunk of size 1001, which is longer than the specified 1000
Created a chunk of size 1463, which is longer than the specified 1000
Created a chunk of size 1703, which is longer than the specified 1000
Created a chunk of size 2118, which is longer than the specified 1000
Created a chunk of size 1348, which is longer than the specified 1000
Created a chunk of size 1149, which is longer than the specified 1000
Created a chunk of s

207 chunks


### Agent as reasoner (Schopenhauer + Ruff)

In [46]:
tools = [
    Tool(
        name="Schopenhauer QA System",
        func=schopenhauer.run,
        description="useful for when you need to answer questions about Arthur Schopenhauer's philosophy in The Wisdom of Life: happiness, personality, wealth, honour, solitude, aphorisms on life. Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
]

agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [47]:
agent.invoke(
    "According to Schopenhauer, what matters most for a person's happiness: what one is, what one has, or how one appears to others?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


To answer the question about Schopenhauer's views on happiness, I will need to reference his philosophy regarding the importance of being, having, and appearance. 

Action: Schopenhauer QA System  
Action Input: According to Schopenhauer, what matters most for a person's happiness: what one is, what one has, or how one appears to others?  
Observation: According to Schopenhauer, what matters most for a person's happiness is what one is. He emphasizes that a person's individuality and inner constitution are the chief factors in their happiness, while external circumstances and how one appears to others are of secondary importance. Schopenhauer argues that true happiness comes from within and is determined by one's own qualities and character rather than by reputation or material possessions.
Thought:I now know the final answer.  
Final Answer: According to Schopenhauer, what matters most for a person's happiness is what one is. He emphasizes that a person's individuality and inner const

{'input': "According to Schopenhauer, what matters most for a person's happiness: what one is, what one has, or how one appears to others?",
 'output': "According to Schopenhauer, what matters most for a person's happiness is what one is. He emphasizes that a person's individuality and inner constitution are the chief factors in their happiness, while external circumstances and how one appears to others are of secondary importance. True happiness comes from within and is determined by one's own qualities and character rather than by reputation or material possessions."}

In [48]:
agent.invoke("Why use ruff over flake8?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to compare the features and benefits of ruff and flake8 to provide a comprehensive answer about why one might choose ruff over flake8. 
Action: Ruff QA System
Action Input: Why should someone use ruff instead of flake8 for Python linting?
Observation: Someone might choose to use Ruff instead of Flake8 for Python linting for several reasons:

1. **Performance**: Ruff is implemented in Rust, which generally allows it to run faster than Flake8, providing quicker feedback on lint violations.

2. **Comprehensive Rule Set**: Ruff implements over 900 rules, significantly more than Flake8, which allows for more extensive linting coverage.

3. **Automatic Fixing**: Unlike Flake8, Ruff is capable of automatically fixing its own lint violations, which can save time during development.

4. **No Custom Rules Needed**: Ruff does not support custom lint rules, but it re-implements popular Flake8 plugins natively, reducing the need for additional plugins and potential conflicts.

5. **Independe

{'input': 'Why use ruff over flake8?',
 'output': 'Someone might choose to use Ruff instead of Flake8 for Python linting due to its performance (being implemented in Rust for faster execution), a comprehensive rule set with over 900 rules, the ability to automatically fix lint violations, independence from plugins, compatibility with type checkers, and ease of use. Ruff provides a more modern and efficient approach to linting, making it an attractive alternative to Flake8 for many developers.'}

In [49]:
agent.invoke(
    "Why use ruff over flake8, and what does Schopenhauer say about the value of solitude?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to address two separate topics: the advantages of using Ruff over Flake8 for Python linting, and Schopenhauer's views on solitude. I will first gather information about Ruff and then about Schopenhauer's philosophy regarding solitude.

Action: Ruff QA System  
Action Input: Why use ruff over flake8?  
Observation: Ruff offers several advantages over Flake8:

1. **Performance**: Ruff is implemented in Rust, which generally allows it to run faster than Flake8, which is written in Python.

2. **More Rules**: Ruff implements over 900 rules, while Flake8 has a smaller set. This means Ruff can catch a wider variety of linting issues.

3. **Automatic Fixes**: Unlike Flake8, Ruff is capable of automatically fixing its own lint violations, which can save time during development.

4. **No Custom Rules**: Ruff does not support custom lint rules, but it re-implements popular Flake8 plugins natively, which can simplify the setup and reduce conflicts.

5. **Independent Plugins**: Ruff uses di

{'input': 'Why use ruff over flake8, and what does Schopenhauer say about the value of solitude?',
 'output': "Ruff is preferred over Flake8 for several reasons: it is faster due to being implemented in Rust, it offers over 900 linting rules, can automatically fix violations, and is designed to work well with type checkers. On the other hand, Schopenhauer values solitude as essential for personal growth, emphasizing that it allows individuals to connect with their true selves and develop their intellectual faculties, contrasting it with the superficiality of social interactions. He believes that solitude is necessary for cultivating one's inner wealth and achieving fulfillment."}

### Agent as router (`return_direct=True`)

In [51]:
direct_tools = [
    Tool(
        name="Schopenhauer QA System",
        func=schopenhauer.run,
        description="useful for when you need to answer questions about Arthur Schopenhauer's philosophy in The Wisdom of Life: happiness, personality, wealth, honour, solitude, aphorisms on life. Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
        return_direct=True,
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
        return_direct=True,
    ),
]

direct_agent = initialize_agent(
    direct_tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [52]:
direct_agent.invoke(
    "According to Schopenhauer, what matters most for a person's happiness: what one is, what one has, or how one appears to others?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


To answer this question, I need to refer to Schopenhauer's philosophy regarding happiness and the factors that contribute to it. I will use the Schopenhauer QA System to get a precise answer.

Action: Schopenhauer QA System  
Action Input: According to Schopenhauer, what matters most for a person's happiness: what one is, what one has, or how one appears to others?  
Observation: According to Schopenhauer, what matters most for a person's happiness is what one is. He emphasizes that a person's individuality and inner constitution are the chief factors in their happiness, while external circumstances and how one appears to others are of secondary importance. Schopenhauer argues that true happiness comes from within and is determined by one's own qualities and character rather than by reputation or material possessions.


> Finished chain.


{'input': "According to Schopenhauer, what matters most for a person's happiness: what one is, what one has, or how one appears to others?",
 'output': "According to Schopenhauer, what matters most for a person's happiness is what one is. He emphasizes that a person's individuality and inner constitution are the chief factors in their happiness, while external circumstances and how one appears to others are of secondary importance. Schopenhauer argues that true happiness comes from within and is determined by one's own qualities and character rather than by reputation or material possessions."}

In [53]:
direct_agent.invoke("Why use ruff over flake8?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to compare the features and benefits of ruff and flake8 to provide a comprehensive answer about why one might choose ruff over flake8. 
Action: Ruff QA System
Action Input: Why should someone use ruff instead of flake8 for Python linting?
Observation: Someone might choose to use Ruff instead of Flake8 for Python linting for several reasons:

1. **Performance**: Ruff is implemented in Rust, which generally allows it to run faster than Flake8, providing quicker feedback on lint violations.

2. **Comprehensive Rule Set**: Ruff implements over 900 rules, significantly more than Flake8, which allows for more extensive linting coverage.

3. **Automatic Fixing**: Unlike Flake8, Ruff is capable of automatically fixing its own lint violations, which can save time during development.

4. **No Custom Rules Needed**: Ruff does not support custom lint rules, but it re-implements popular Flake8 plugins natively, reducing the need for additional plugins and potential conflicts.

5. **Independe

{'input': 'Why use ruff over flake8?',
 'output': 'Someone might choose to use Ruff instead of Flake8 for Python linting for several reasons:\n\n1. **Performance**: Ruff is implemented in Rust, which generally allows it to run faster than Flake8, providing quicker feedback on lint violations.\n\n2. **Comprehensive Rule Set**: Ruff implements over 900 rules, significantly more than Flake8, which allows for more extensive linting coverage.\n\n3. **Automatic Fixing**: Unlike Flake8, Ruff is capable of automatically fixing its own lint violations, which can save time during development.\n\n4. **No Custom Rules Needed**: Ruff does not support custom lint rules, but it re-implements popular Flake8 plugins natively, reducing the need for additional plugins and potential conflicts.\n\n5. **Independence from Plugins**: Ruff includes all rules natively, while Flake8 relies on plugins (checkers) that can introduce complexity and compatibility issues.\n\n6. **Support for Multiple Tools**: Ruff can

In [54]:
direct_agent.invoke(
    "Why use ruff over flake8, and what does Schopenhauer say about the value of solitude?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to address two separate topics: the advantages of using Ruff over Flake8, and Schopenhauer's views on solitude. I will first gather information about Ruff and Flake8, then I will look into Schopenhauer's philosophy regarding solitude.

Action: Ruff QA System  
Action Input: What are the advantages of using Ruff over Flake8?  
Observation: The advantages of using Ruff over Flake8 include:

1. **Performance**: Ruff is implemented in Rust, which generally allows it to run faster than Flake8, providing quicker feedback on lint violations.

2. **Comprehensive Rule Set**: Ruff implements over 900 rules, significantly more than Flake8, which allows for more extensive linting capabilities.

3. **Automatic Fixing**: Unlike Flake8, Ruff is capable of automatically fixing its own lint violations, which can save time during development.

4. **No Custom Rules**: Ruff does not support custom lint rules, but it re-implements popular Flake8 plugins natively, reducing the complexity of managing 

{'input': 'Why use ruff over flake8, and what does Schopenhauer say about the value of solitude?',
 'output': 'The advantages of using Ruff over Flake8 include:\n\n1. **Performance**: Ruff is implemented in Rust, which generally allows it to run faster than Flake8, providing quicker feedback on lint violations.\n\n2. **Comprehensive Rule Set**: Ruff implements over 900 rules, significantly more than Flake8, which allows for more extensive linting capabilities.\n\n3. **Automatic Fixing**: Unlike Flake8, Ruff is capable of automatically fixing its own lint violations, which can save time during development.\n\n4. **No Custom Rules**: Ruff does not support custom lint rules, but it re-implements popular Flake8 plugins natively, reducing the complexity of managing multiple plugins.\n\n5. **Simplified Rule Selection**: Ruff uses different rule codes and prefixes to minimize conflicts across plugins, allowing users to toggle rules on or off more easily.\n\n6. **Independence from Plugins**: R

## Reflection

- **Schopenhauer-only question** (what matters most for happiness): the agent chose the *Schopenhauer QA System* both times and answered that it is *what one is* (personality) that matters most, not what one has or how one appears.
- **Ruff-only question** (why use ruff over flake8): the agent chose the *Ruff QA System* both times. The tool descriptions were enough to route it correctly.
- **Multi-hop question** (ruff vs flake8 *and* Schopenhauer on solitude): with the normal agent (the reasoner), it called the Ruff tool, then the Schopenhauer tool, and combined both results in one final answer.
- **`return_direct=True`**: the agent returned the first tool's raw answer immediately, with no extra reasoning step. That is faster and fine for single-source questions, but on the multi-hop question it stopped after one tool and only answered the Ruff half. For questions that need several sources, the reasoner agent (`return_direct=False`) works better.